# 02 - Data Cleaning & Preparation
IPL Match & Player Performance Analysis

Input: `data/raw/` (never edited). Output: `data/cleaned/` + a cleaning log.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
CLEAN = Path("../data/cleaned")
CLEAN.mkdir(parents=True, exist_ok=True)

matches = pd.read_csv(RAW / "matches.csv")
deliveries = pd.read_csv(RAW / "deliveries.csv")

rows_before = {"matches": len(matches), "deliveries": len(deliveries)}
print("BEFORE:", rows_before)

log = []
def add_log(table, issue, action, reason, count):
    log.append({"table": table, "issue": issue, "action": action,
                "reason": reason, "values_affected": int(count)})

BEFORE: {'matches': 1095, 'deliveries': 260920}


## 1. Check for duplicates

In [2]:
dup_m = matches.duplicated().sum()
dup_d = deliveries.duplicated().sum()
matches = matches.drop_duplicates()
deliveries = deliveries.drop_duplicates()
add_log("matches", "Duplicate rows", "Checked and removed", "Duplicates would double-count wins", dup_m)
add_log("deliveries", "Duplicate rows", "Checked and removed", "Duplicates would double-count runs", dup_d)
print("Duplicates found:", dup_m, dup_d)

Duplicates found: 0 0


## 2. Standardise team names
Same franchise, different names over the years. (Deccan Chargers and Gujarat Lions are different teams, so they stay separate.)

In [3]:
team_map = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

fixed = 0
for col in ["team1", "team2", "toss_winner", "winner"]:
    fixed += matches[col].isin(team_map).sum()
    matches[col] = matches[col].replace(team_map)
for col in ["batting_team", "bowling_team"]:
    fixed += deliveries[col].isin(team_map).sum()
    deliveries[col] = deliveries[col].replace(team_map)

add_log("both", "Inconsistent team names (renamed franchises)", "Mapped old names to current names",
        "Same franchise must be counted as one team", fixed)
print("Teams now:", sorted(set(matches.team1) | set(matches.team2)))

Teams now: ['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


## 3. Standardise venue and city names
58 venue names are really about 35 stadiums (same ground written differently, or renamed).

In [4]:
venue_map = {
    "Arun Jaitley Stadium, Delhi": "Arun Jaitley Stadium",
    "Feroz Shah Kotla": "Arun Jaitley Stadium",
    "Brabourne Stadium, Mumbai": "Brabourne Stadium",
    "Dr DY Patil Sports Academy, Mumbai": "Dr DY Patil Sports Academy",
    "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam": "Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium",
    "Eden Gardens, Kolkata": "Eden Gardens",
    "Himachal Pradesh Cricket Association Stadium, Dharamsala": "Himachal Pradesh Cricket Association Stadium",
    "M Chinnaswamy Stadium, Bengaluru": "M Chinnaswamy Stadium",
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "MA Chidambaram Stadium, Chepauk": "MA Chidambaram Stadium",
    "MA Chidambaram Stadium, Chepauk, Chennai": "MA Chidambaram Stadium",
    "Maharashtra Cricket Association Stadium, Pune": "Maharashtra Cricket Association Stadium",
    "Subrata Roy Sahara Stadium": "Maharashtra Cricket Association Stadium",
    "Narendra Modi Stadium, Ahmedabad": "Narendra Modi Stadium",
    "Sardar Patel Stadium, Motera": "Narendra Modi Stadium",
    "Punjab Cricket Association IS Bindra Stadium, Mohali": "Punjab Cricket Association IS Bindra Stadium",
    "Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh": "Punjab Cricket Association IS Bindra Stadium",
    "Punjab Cricket Association Stadium, Mohali": "Punjab Cricket Association IS Bindra Stadium",
    "Rajiv Gandhi International Stadium, Uppal": "Rajiv Gandhi International Stadium",
    "Rajiv Gandhi International Stadium, Uppal, Hyderabad": "Rajiv Gandhi International Stadium",
    "Sawai Mansingh Stadium, Jaipur": "Sawai Mansingh Stadium",
    "Zayed Cricket Stadium, Abu Dhabi": "Sheikh Zayed Stadium",
    "Wankhede Stadium, Mumbai": "Wankhede Stadium",
}
n_before = matches.venue.nunique()
fixed = matches.venue.isin(venue_map).sum()
matches["venue"] = matches["venue"].replace(venue_map)
add_log("matches", f"Duplicate venue names ({n_before} names)", f"Merged into {matches.venue.nunique()} stadiums",
        "Wins at one stadium were split across several names", fixed)
print("Venues:", n_before, "->", matches.venue.nunique())

# City: fix spelling, then fill the 51 missing cities from the venue
fixed = matches.city.isin(["Bangalore", "Chandigarh"]).sum()
matches["city"] = matches["city"].replace({"Bangalore": "Bengaluru", "Chandigarh": "Mohali"})
add_log("matches", "City spelling (Bangalore/Bengaluru, Chandigarh/Mohali)", "Standardised",
        "Same city under two names", fixed)

missing_city = matches.city.isna().sum()
matches["city"] = matches["city"].fillna(matches["venue"].map({
    "Dubai International Cricket Stadium": "Dubai",
    "Sharjah Cricket Stadium": "Sharjah"}))
add_log("matches", "Missing city", "Filled from venue (Dubai, Sharjah)",
        "Venue name uniquely identifies the city", missing_city - matches.city.isna().sum())
print("Missing cities left:", matches.city.isna().sum())

Venues: 58 -> 36
Missing cities left: 0


## 4. Fix dates and season
Season has mixed formats (`2007/08`, `2009`, `2020/21`). Use the actual year of the match instead.

In [5]:
matches["date"] = pd.to_datetime(matches["date"])
old_season = matches["season"].astype(str)
matches["season"] = matches["date"].dt.year
add_log("matches", "Mixed season format (2007/08, 2009/10, 2020/21 vs plain years)", "Season = year of match date",
        "One consistent numeric season for grouping", (old_season != matches["season"].astype(str)).sum())
print(sorted(matches.season.unique()))

[np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]


## 5. Handle nulls in `matches`
Most nulls here are not errors. They mean no result, tie, or no rain rule used.

In [6]:
print(matches.isna().sum()[lambda s: s > 0])

# 5 abandoned matches: no winner, no player of match
n = matches.winner.isna().sum()
matches["winner"] = matches["winner"].fillna("No Result")
matches["player_of_match"] = matches["player_of_match"].fillna("No Award")
add_log("matches", "Missing winner / player_of_match", "Filled with 'No Result' / 'No Award'",
        "Match was abandoned - null is real information", n)

# method: only filled when Duckworth-Lewis was used
n = matches.method.isna().sum()
matches["method"] = matches["method"].fillna("Normal")
add_log("matches", "Missing method", "Filled with 'Normal'", "Empty means no D/L adjustment", n)

# result_margin / target: left as null on purpose (ties and abandoned matches have no margin)
add_log("matches", "Missing result_margin (19) and target_runs (3)", "Left as null",
        "Ties and no-result matches have no margin; filling with 0 would distort averages",
        matches.result_margin.isna().sum() + matches.target_runs.isna().sum())

# Wins need a flag that excludes no-result matches
matches["is_completed"] = (matches["result"] != "no result").astype(int)

player_of_match       5
winner                5
result_margin        19
target_runs           3
target_overs          3
method             1074
dtype: int64


## 6. Clean `deliveries`

In [7]:
# extras_type is empty when there was no extra
n = deliveries.extras_type.isna().sum()
deliveries["extras_type"] = deliveries["extras_type"].fillna("none")
add_log("deliveries", "Missing extras_type", "Filled with 'none'", "Empty means a normal delivery", n)

# player_dismissed / dismissal_kind / fielder stay null when no wicket (null = no dismissal)
add_log("deliveries", "Missing player_dismissed, dismissal_kind, fielder", "Left as null",
        "Null means no wicket on that ball (the raw file wrote this as text 'NA')",
        deliveries.player_dismissed.isna().sum())

# Derived columns needed for fair cricket statistics
deliveries["is_super_over"] = (deliveries["inning"] > 2).astype(int)

# Wides and no-balls are not legal deliveries (bowler's over doesn't count them)
deliveries["is_legal_ball"] = (~deliveries["extras_type"].isin(["wides", "noballs"])).astype(int)

# A batter does not face a wide
deliveries["ball_faced"] = (deliveries["extras_type"] != "wides").astype(int)

# Byes/leg byes are not charged to the bowler
deliveries["bowler_runs"] = np.where(deliveries["extras_type"].isin(["byes", "legbyes", "penalty"]),
                                     0, deliveries["total_runs"])

# Run outs etc. are not credited to the bowler
not_bowler = ["run out", "retired hurt", "retired out", "obstructing the field"]
deliveries["bowler_wicket"] = ((deliveries["is_wicket"] == 1) &
                               (~deliveries["dismissal_kind"].isin(not_bowler))).astype(int)

add_log("deliveries", "No columns for legal balls, balls faced, bowler runs, bowler wickets",
        "Created is_super_over, is_legal_ball, ball_faced, bowler_runs, bowler_wicket",
        "Needed for correct strike rate, economy and wicket counts", len(deliveries))

## 7. Validation and outlier checks

In [8]:
# Do runs add up?
assert (deliveries.batsman_runs + deliveries.extra_runs == deliveries.total_runs).all()

# Every delivery has a match, every match has deliveries
assert set(deliveries.match_id) == set(matches.id)

# Outliers: cricket scores are valid even when extreme, so we check but do not delete
print("Max runs on one ball:", deliveries.total_runs.max())
print("Target runs range:", matches.target_runs.min(), "-", matches.target_runs.max())
print("Result margin (runs) max:", matches.loc[matches.result == "runs", "result_margin"].max())
add_log("both", "Outliers (very high targets / big margins)", "Checked, kept",
        "Genuine cricket results, e.g. 287-run total and 146-run win", 0)

# Data types
matches["id"] = matches["id"].astype(int)
print(matches.dtypes)

Max runs on one ball: 7
Target runs range: 43.0 - 288.0
Result margin (runs) max: 146.0
id                          int64
season                      int32
city                          str
date               datetime64[us]
match_type                    str
player_of_match               str
venue                         str
team1                         str
team2                         str
toss_winner                   str
toss_decision                 str
winner                        str
result                        str
result_margin             float64
target_runs               float64
target_overs              float64
super_over                    str
method                        str
umpire1                       str
umpire2                       str
is_completed                int64
dtype: object


## 8. Build `teams` and `players` tables

In [9]:
teams = pd.DataFrame({"team_name": sorted(set(matches.team1) | set(matches.team2))})
teams.insert(0, "team_id", range(1, len(teams) + 1))

all_names = pd.concat([deliveries.batter, deliveries.bowler, deliveries.non_striker,
                       matches.player_of_match[matches.player_of_match != "No Award"]])
players = pd.DataFrame({"player_name": sorted(all_names.dropna().unique())})
players.insert(0, "player_id", range(1, len(players) + 1))

add_log("new tables", "No teams or players table in source", "Created teams and players",
        "Needed for 3-4 table SQL joins", len(teams) + len(players))
print(len(teams), "teams,", len(players), "players")

15 teams, 732 players


## 9. Save cleaned files and show before/after

In [10]:
matches.to_csv(CLEAN / "matches_clean.csv", index=False)
deliveries.to_csv(CLEAN / "deliveries_clean.csv", index=False)
teams.to_csv(CLEAN / "teams.csv", index=False)
players.to_csv(CLEAN / "players.csv", index=False)

log_df = pd.DataFrame(log)
log_df.to_csv(CLEAN / "cleaning_log.csv", index=False)

summary = pd.DataFrame({
    "table": ["matches", "deliveries"],
    "rows_before": [rows_before["matches"], rows_before["deliveries"]],
    "rows_after": [len(matches), len(deliveries)],
})
print(summary)
log_df

        table  rows_before  rows_after
0     matches         1095        1095
1  deliveries       260920      260920


,table,issue,action,reason,values_affected
0,matches,Duplicate rows,Checked and removed,Duplicates would double-count wins,0
1,deliveries,Duplicate rows,Checked and removed,Duplicates would double-count runs,0
2,both,Inconsistent team names (renamed franchises),Mapped old names to current names,Same franchise must be counted as one team,143564
3,matches,Duplicate venue names (58 names),Merged into 36 stadiums,Wins at one stadium were split across several ...,481
4,matches,"City spelling (Bangalore/Bengaluru, Chandigarh...",Standardised,Same city under two names,126
5,matches,Missing city,"Filled from venue (Dubai, Sharjah)",Venue name uniquely identifies the city,51
6,matches,"Mixed season format (2007/08, 2009/10, 2020/21...",Season = year of match date,One consistent numeric season for grouping,178
7,matches,Missing winner / player_of_match,Filled with 'No Result' / 'No Award',Match was abandoned - null is real information,5
8,matches,Missing method,Filled with 'Normal',Empty means no D/L adjustment,1074
9,matches,Missing result_margin (19) and target_runs (3),Left as null,Ties and no-result matches have no margin; fil...,22
